# Canonical Model 03: PRT and Parallel Splitting

Build and run the 50 x 50, four-layer canonical DISV model, trace particles with MF6 PRT, and split the same simulation across eight real MPI processes.

> **Validation goal:** every stage must terminate normally, preserve both physical lakes, and produce reconstructable results.


In [1]:
from pathlib import Path
import pandas as pd
import simple_modflow as mf
from simple_modflow.modflow.mf6.canonical_example import representative_cells
from canonical_notebook_style import notebook_header

notebook_header('03', 'PRT and Parallel', 'Trace groundwater movement and run the same model across MPI partitions.')

root = Path('../artifacts/canonical_prt_parallel')
config = mf.CanonicalModelConfig.validation()
model = mf.build_canonical_model(root / 'gwf', config=config)
success, gwf_report = model.run_simulation()
assert success, '\n'.join(gwf_report[-30:])

pd.Series({
    'rows': config.nrow,
    'columns': config.ncol,
    'layers': config.nlay,
    'total_3d_cells': config.nrow * config.ncol * config.nlay,
    'gwf_termination': gwf_report[-1],
}, name='canonical GWF')


VoronoiGrid initializing.
Voronoi grid initialized.
Imported 1 features from ..\artifacts\canonical_prt_parallel\gwf\inputs\north_lake.gpkg
Imported 1 features from ..\artifacts\canonical_prt_parallel\gwf\inputs\south_lake.gpkg
Generating connections (rectangular mode) for lake 0
getting connectivity properties (iac, ja, cl12, hwva, nja)
Generating connections (rectangular mode) for lake 1


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:302: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model viz_prt_master...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 200 based on size of stress_period_data
    writing package ghb...
INFORMATION: maxbound in ('', 'ghb', 'dimensions') changed to 200 based on size of stress_period_data
    writing package rch...
    writing package wel...
INFORMATION: maxbound in ('', 'wel', 'dimensions') changed to 2 based on size of stress_period_data
    writing package drn...
INFORMATION: maxbound in ('', 'drn', 'dimensions') changed to 124 based on size of stress_period_data
    writing package lak...
    writing package sfr...
    writing package mvr...
    writing package uzf...
    writing package gwf_obs...
    writing pack

rows                                               50
columns                                            50
layers                                              4
total_3d_cells                                  10000
gwf_termination     Normal termination of simulation.
Name: canonical GWF, dtype: object

## MF6 PRT

PRT consumes the completed GWF head, budget, and binary-grid files. Tracking stops at the end of available flow output by default, preventing an unbounded final-time-step run.


In [ ]:
releases = mf.PRTReleasePoints.from_cells(model, representative_cells(config)['releases'])
prt = model.particle_tracking.prt(
    workspace=root / 'prt',
    release_points=releases,
    porosity=0.25,
    extend_tracking=False,
)
prt_results = prt.run(silent=False)
assert prt_results.success
assert not prt_results.pathlines.empty

pd.Series({
    'release_points': len(releases.packagedata),
    'pathline_records': len(prt_results.pathlines),
    'tracked_particles': prt_results.pathlines['irpt'].nunique(),
    'track_csv_bytes': prt_results.track_csv_path.stat().st_size,
}, name='PRT results')


In [ ]:
import pandas as pd
import figs as f

model.gwf.npf.k[0].get_data().reshape((50,50))

fig = f.Fig()

fig.add_heatmap(z=model.gwf.npf.k[0].get_data().reshape((50,50)))

fig.show()

In [ ]:
model.modelgrid.ncpl

In [ ]:
model.hds.map().plot()

In [ ]:
model.particle_tracking.open_prt(workspace=r"C:\Users\lukem\Python\Projects\simple_modflow\examples\mf6\artifacts\canonical_prt_parallel\prt").scene().export_html(path=r"C:\Users\lukem\Python\Projects\simple_modflow\examples\mf6\artifacts\canonical_prt_parallel\prt\prt.html")

## Lake-Safe Partitions

Partition cuts route around each physical lake. This avoids creating zero-area local LAK fragments while retaining contiguous model partitions.


In [2]:
partition_rows = []
for nparts in range(2, 9):
    candidate_mask = mf.canonical_partition_mask(model, nparts)
    print(f'Candidate partition mask: {candidate_mask.shape}')
    prepared = model.parallel.split_model(
        workspace=root / f'split_{nparts}',
        mask=candidate_mask,
        write=False,
    )
    print(f'Prepared split: {nparts}')
    validation = prepared.validate()
    partition_rows.append({
        'partitions': nparts,
        'all_contiguous': bool(validation['contiguous'].all()),
        'minimum_columns': int(validation['columns'].min()),
        'maximum_columns': int(validation['columns'].max()),
    })
    print(f'Prepared and validated {nparts} contiguous partitions.')

partition_summary = pd.DataFrame(partition_rows).set_index('partitions')
assert partition_summary['all_contiguous'].all()
partition_summary


Candidate partition mask: (2500,)
Prepared split: 2
Prepared and validated 2 contiguous partitions.
Candidate partition mask: (2500,)
Prepared split: 3
Prepared and validated 3 contiguous partitions.
Candidate partition mask: (2500,)
Prepared split: 4
Prepared and validated 4 contiguous partitions.
Candidate partition mask: (2500,)
Prepared split: 5
Prepared and validated 5 contiguous partitions.
Candidate partition mask: (2500,)
Prepared split: 6
Prepared and validated 6 contiguous partitions.
Candidate partition mask: (2500,)
Prepared split: 7
Prepared and validated 7 contiguous partitions.
Candidate partition mask: (2500,)
Prepared split: 8
Prepared and validated 8 contiguous partitions.


,all_contiguous,minimum_columns,maximum_columns
partitions,,,
2,True,1250,1250
3,True,752,914
4,True,302,948
5,True,236,784
6,True,202,712
7,True,86,806
8,True,74,732


In [3]:
from pathlib import Path
from time import perf_counter

import numpy as np
from flopy.mf6.utils import Mf6Splitter

from simple_modflow.modflow.mf6.parallel import (
    ParallelSplitRun,
    _preserve_lake_partitions,
    _repair_split_lak_packages,
    _source_without_observation_packages,
)

workspace = Path("split_profile")
nparts = 8
timings = {}


def measure(name, function):
    start = perf_counter()
    result = function()
    timings[name] = perf_counter() - start
    print(f"{name:<35} {timings[name]:8.3f} seconds")
    return result


mask = measure(
    "Create partition mask",
    lambda: mf.canonical_partition_mask(model, nparts),
)

splitter = measure(
    "Initialize FloPy splitter",
    lambda: Mf6Splitter(model.sim),
)

mask = measure(
    "Preserve lakes and MVR topology",
    lambda: _preserve_lake_partitions(model.gwf, mask),
)

guard = _source_without_observation_packages(model.gwf)

start = perf_counter()
guard.__enter__()
timings["Source-data guard"] = perf_counter() - start

try:
    split_simulation = measure(
        "FloPy split_model remapping",
        lambda: splitter.split_model(mask),
    )
finally:
    guard.__exit__(None, None, None)

measure(
    "Repair split LAK packages",
    lambda: _repair_split_lak_packages(split_simulation),
)

split = ParallelSplitRun(
    model,
    splitter,
    split_simulation,
    np.asarray(mask),
    workspace,
)

measure("Validate partitions", split.validate)
measure("Write split workspace", split.write)

Create partition mask                  0.093 seconds
Initialize FloPy splitter              0.000 seconds
Preserve lakes and MVR topology        0.013 seconds
FloPy split_model remapping           90.676 seconds
Repair split LAK packages              0.012 seconds
Validate partitions                    0.112 seconds
writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing package viz_prt_master_0_viz_prt_master_1...
  writing package viz_prt_master_0_viz_prt_master_1.mvr...
  writing package viz_prt_master_1_viz_prt_master_2...
  writing package viz_prt_master_1_viz_prt_master_2.mvr...
  writing package viz_prt_master_2_viz_prt_master_3...
  writing package viz_prt_master_2_viz_prt_master_3.mvr...
  writing package viz_prt_master_3_viz_prt_master_4...
  writing package viz_prt_master_3_viz_prt_master_4.mvr...
  writing package viz_prt_master_4_viz_prt_master_5...
  writing package viz_prt_master_4_viz_pr

WindowsPath('split_profile')

In [4]:
from time import perf_counter
from flopy.mf6.utils import Mf6Splitter

mask = mf.canonical_partition_mask(model, 8)

splitter = Mf6Splitter(model.sim)

start = perf_counter()
split_simulation = splitter.split_model(mask)  # Direct FloPy call
print(f"FloPy alone: {perf_counter() - start:.2f} seconds")

KeyError: 'STAGE'

## Eight-Process MPI Run

The final partition set is written and run with eight MPI workers. The current Python environment's matching mf6 and mpiexec executables are discovered automatically.


In [ ]:
mask = mf.canonical_partition_mask(model, 8)
split = model.parallel.split_model(workspace=root / 'split_8', mask=mask)
split.summary()


In [ ]:
parallel_success, parallel_report = split.run(processors=split.nparts, write=False, silent=False)
assert parallel_success, '\n'.join(parallel_report[-30:])
assert any('PARALLEL mode' in line for line in parallel_report)

pd.Series({
    'processors': split.nparts,
    'parallel_mode_confirmed': True,
    'termination': parallel_report[-1],
}, name='MPI run')


## Reconstructed Results

Split outputs are reconstructed onto the original DISV grid and compared with the source-model heads. Small numerical differences are expected; large differences indicate a split or exchange problem.


In [ ]:
head_comparison = split.compare_heads()
assert head_comparison.loc[0, 'max_absolute_error'] < 0.1, head_comparison
head_comparison


## Result

The same 50 x 50 canonical model now completes as a source GWF simulation, a finite MF6 PRT simulation, and an eight-process MPI split simulation. Every code-cell output above is part of the validation record.
